In [1]:
!pip install groq -q
print("Library installed successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.2 MB/s eta 0:00:00
Library installed successfully


In [2]:
import pandas as pd
import os
import sqlite3
from groq import Groq

In [3]:
os.environ["GROQ_API_KEY"] = "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"

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"

print("Groq client initialized successfully")
print(f"Using Model: {MODEL}")

Groq client initialized successfully
Using Model: llama-3.1-8b-instant


In [4]:
import io
csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""

df=pd.read_csv(io.StringIO(csv_data))
print(f"Dataset Loaded:{len(df)} rows, {len(df.columns)} columns")
print("\nFirst 5 rows:")
df.head()

Dataset Loaded:30 rows, 8 columns

First 5 rows:


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [5]:
conn=sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database Created: college.db")
print("Table 'students' crerated with 30 student records" )

test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f"\nVerification: {test_df['total_rows'][0]} rows in database")

Database Created: college.db
Table 'students' crerated with 30 student records

Verification: 30 rows in database


In [6]:
def get_schema(conn,table_name='students'):
  """
  This Function reads the structure of the database table.
  It returns information about each column name and data type

  Returns:
     A formatted string describing the table structure
  """

  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns=cursor.fetchall()

  schema_lines=[f"Table:{table_name}"]
  schema_lines.append("columns:")

  for col in columns:
    schema_lines.append(f" - {col[1]}({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")

  sample_rows=cursor.fetchall()
  schema_lines.append("\n Sample rows (first 3):")

  for row in sample_rows:
    schema_lines.append(f" {row}")

  return "\n".join(schema_lines)

schema=get_schema(conn)
print(schema)

Table:students
columns:
 - student_id(INTEGER)
 - name(TEXT)
 - age(INTEGER)
 - gender(TEXT)
 - subject(TEXT)
 - marks(INTEGER)
 - attendance(INTEGER)
 - grade(TEXT)

 Sample rows (first 3):
 (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
 (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
 (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [7]:
def generate_sql(user_question,schema_text,client,model):
  """
  Send's to the user's question and database schema to the Groq LLM
  LLM generates the SQL query that answers the question
 """

  system_prompt=f"""You are an expert SQL assistant.
 You are connected to a SQLite database with the following structure:
 {schema_text}

 Rules to follow:
 1.Generate only a valid SQLite SQL query.
 2.Do not include any explanatio or text.

 3.Do not use markdown code blocks. Return only the raw SQL query.
 4.Table name is :students
 5.Only use the column names that exists in the schema above
 6.Use single quotes for strting values in WHERE clauses
 7.If the user asks for a top N, use ORDER BY marks DESC LIMIT N.
 """

  response=client.chat.completions.create(
      model=model,
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user_question}
      ],
      temperature=0.0
  )

  sql_query=response.choices[0].message.content.strip()
  return sql_query


question="Show me all female students"
print(f"Question: {question}")
print("\nGenerating SQL Query...")
sql_query=generate_sql(question,schema,client,MODEL)
print(f"\nSQL Query: \n{sql_query}")

Question: Show me all female students

Generating SQL Query...

SQL Query: 
SELECT * FROM students WHERE gender = 'Female'


In [8]:
import re

def execute_sql(sql_query,conn):
  clean_sql=sql_query.strip()
  clean_sql=re.sub(r'```sql\s*','',clean_sql)
  clean_sql=clean_sql.strip()
  clean_sql=re.sub(r'```\s*','',clean_sql)
  clean_sql=clean_sql.strip()
  try:
    result_df=pd.read_sql_query(clean_sql,conn)
    return result_df,None
  except Exception as e:
    return None, str(e)
print(f"Executing SQL: {sql_query}")
result,error = execute_sql(sql_query,conn)
if error:
  print(f"Error: {error}")
else:
  print(f"\nQuery returned {len(result)} rows:")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Female'

Query returned 15 rows:
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female 

In [9]:
def text_to_sql_agent(user_question,conn,client,model,verbose=True):
  print("="*60)
  print(f"User Question: {user_question}")
  print("="*60)
  if verbose:
    print("\n[STEP 1] Reading database schema...")
  schema_text = get_schema(conn)
  if verbose:
    print("Schema loaded successfully")
  if verbose:
    print("[STEP 2] Generating SQL query with Groq LLM...")
  generated_sql = generate_sql(user_question,schema_text,client,model)
  if verbose:
    print(f"Generated SQL:\n {generated_sql}")
  if verbose:
    print("\n[STEP 3] Executing SQL on the database...")
  result_df,error=execute_sql(generated_sql,conn)
  if error:
    print(f"SQL Execution Error: {error}")
    return None, generated_sql
  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-"*40)
    print(result_df.to_string(index=False))
  print("="*60)
  return result_df,generated_sql
result,sql_used=text_to_sql_agent(
    "Show top 5 studnets in Programnming",
    conn,client,MODEL
)

User Question: Show top 5 studnets in Programnming

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
         11 Aditya Kumar   21   Male Programming     97          99    A+
          3  Rohan Mehta   20   Male Programming     95          98    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
          5   Arjun Nair   21   Male Programming     91          94    A+


In [10]:
result1, _=text_to_sql_agent(
    "Show me all students who study Mathematics",
    conn,client,MODEL
)

User Question: Show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT * FROM students WHERE subject = 'Mathematics'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fe

In [11]:
result2, _=text_to_sql_agent(
    "What is the average mark for each subject?",
    conn,client,MODEL
)

User Question: What is the average mark for each subject?

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT subject, AVG(marks) FROM students GROUP BY subject

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [12]:
result3,_=text_to_sql_agent(
    "How many students are there in each grade?",
    conn,client,MODEL
)

User Question: How many students are there in each grade?

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT grade, COUNT(*) FROM students GROUP BY grade

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
grade  COUNT(*)
    A         9
   A+         7
    B         8
    C         4
    D         2


In [13]:
result4,_=text_to_sql_agent(
    "What is the average age of male students?",
    conn,client,MODEL
)


User Question: What is the average age of male students?

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT AVG(age) FROM students WHERE gender = 'Male'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 AVG(age)
20.866667


In [14]:
result5,_=text_to_sql_agent(
    "What is the total number of male students?",
    conn,client,MODEL
)


User Question: What is the total number of male students?

[STEP 1] Reading database schema...
Schema loaded successfully
[STEP 2] Generating SQL query with Groq LLM...
Generated SQL:
 SELECT COUNT(student_id) FROM students WHERE gender = 'Male'

[STEP 3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 COUNT(student_id)
                15


In [15]:
def generate_answwer(user_question,query_results_df,client,model):
  if query_results_df is None or len(query_results_df)==0:
    return "No results were found for your query"
  results_text=query_results_df.to_string(index=False)
  prompt=f"""The user asked : '{user_question}'
  The database returned these results:
  {results_text}
  Please write a clear, friendly, 2-3 sentence answer to the users question based on these results.
  Be specific. Mention actual names and numbers from data.
  Do not add information not present in the results."""
  response=client.chat.completions.create(
      model=model,
      messages=[{'role':'user','content':prompt}],
      temperature=0.3
  )
  return response.choices[0].message.content.strip()

In [19]:
def start_text_to_sql_agent(user_question,conn,client,model):
  print('='*60)
  print(f"User Question: {user_question}")
  print('='*60)

  schema_test=get_schema(conn)
  print("Generating SQL...")
  generated_sql=generate_sql(user_question,schema_test,client,model)
  print(f" SQL:\n{generated_sql}")

  result_df,error=execute_sql(generated_sql,conn)
  if error:
      print(f"Error executing SQL:{error}")
      return

  print(f"\nData ({len(result_df)}) rows returned")
  display(result_df)

  print("\nGenerating natural language answer...")
  answer=generate_answwer(user_question,result_df,client,model)
  print(f"\nAnswer: {answer}")
  print('='*60)
  return answer

def smart_text_to_sql_agent(user_question, conn, client, model):
  print("="*60)
  print(f"User Question: {user_question}")
  print("="*60)

  print("\n[STEP 1] Reading database schema...")
  schema_text = get_schema(conn)
  print("Schema loaded successfully")

  print("[STEP 2] Generating SQL query with Groq LLM...")
  generated_sql = generate_sql(user_question, schema_text, client, model)
  print(f"Generated SQL:\n {generated_sql}")

  print("\n[STEP 3] Executing SQL on the database...")
  result_df, error = execute_sql(generated_sql, conn)
  if error:
    print(f"SQL Execution Error: {error}")
    return None, generated_sql

  print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
  print("\nRESULTS:")
  print("-"*40)
  display(result_df)

  print("\n[STEP 5] Generating natural language answer...")
  answer = generate_answwer(user_question, result_df, client, model)
  print(f"\nAnswer: {answer}")
  print("="*60)
  return result_df, generated_sql

# Demonstrate the use of start_text_to_sql_agent
start_text_to_sql_agent(
    "What is the average attendance for female students?",
    conn, client, MODEL
)

User Question: What is the average attendance for female students?
Generating SQL...
 SQL:
SELECT AVG(attendance) FROM students WHERE gender = 'Female'

Data (1) rows returned


,AVG(attendance)
0,87.8



Generating natural language answer...

Answer: Based on the data, the average attendance for female students is 87.8. This means that, on average, female students have been attending classes at a rate of 87.8% throughout the semester.


'Based on the data, the average attendance for female students is 87.8. This means that, on average, female students have been attending classes at a rate of 87.8% throughout the semester.'